## ライブラリの読み込み

In [1]:
import random
from beamngpy import BeamNGpy, Scenario, Vehicle, set_up_simple_logging

## BeamNGの起動

In [2]:
random.seed(1703)
set_up_simple_logging()

beamng = BeamNGpy('localhost', 64256)
bng = beamng.open(launch=False)

2025-11-15 15:38:03,717 |INFO     |beamngpy                      |Started BeamNGpy logging.
2025-11-15 15:38:04,120 |INFO     |beamngpy.BeamNGpy             |Successfully connected to BeamNG.tech.
2025-11-15 15:38:04,121 |INFO     |beamngpy.BeamNGpy             |BeamNGpy successfully connected to existing BeamNG instance.


## Map

In [3]:
for level in bng.get_levels().keys():
  print(level)

2k_tsukuba
2k_tsukuba_s
automation_test_track
autotest
c1
Cliff
derby
driver_training
east_coast_usa
garage_v2
glow_city
gridmap_v2
hirochi_raceway
Industrial
italy
johnson_valley
jungle_rock_island
showroom_v2
small_island
smallgrid
tech_ground
template
Utah
west_coast_usa


## Vehicle

### 使用できる車両一覧

In [4]:
# 利用可能な車両一覧を取得
available_vehicles = bng.vehicles.get_available()

# 実際の車両データを取得
vehicles_data = available_vehicles['vehicles']

# 車両タイプごとに分類（CarとTruckのみ）
vehicle_types = {}
for key, vehicle_data in vehicles_data.items():
    vehicle_type = vehicle_data.get('type', 'Unknown')
    
    # CarとTruckのみをフィルタリング
    if vehicle_type in ['Car', 'Truck']:
        if vehicle_type not in vehicle_types:
            vehicle_types[vehicle_type] = []
        vehicle_types[vehicle_type].append({
            'key': key,
            'name': vehicle_data.get('name', 'Unknown'),
            'author': vehicle_data.get('author', 'Unknown'),
            'config_count': len(vehicle_data.get('configurations', {})),
            'configurations': vehicle_data.get('configurations', {})
        })

# 見やすく表示
print("=" * 80)
print("BeamNG.tech 利用可能な車両一覧 (Car & Truck)")
print("=" * 80)

for vehicle_type in sorted(vehicle_types.keys()):
    print(f"\n【{vehicle_type}】 ({len(vehicle_types[vehicle_type])}台)")
    print("-" * 80)
    
    for idx, veh in enumerate(sorted(vehicle_types[vehicle_type], key=lambda x: x['name']), 1):
        print(f"  {idx:3d}. {veh['name']:<40} (key: {veh['key']})")
        print(f"       設定数: {veh['config_count']}, 作者: {veh['author']}")

print("\n" + "=" * 80)
print(f"総車両数 (Car & Truck): {sum(len(v) for v in vehicle_types.values())}")
print("=" * 80)

data: {'type': 'GetAvailableVehicles'}
BeamNG.tech 利用可能な車両一覧 (Car & Truck)

【Car】 (28台)
--------------------------------------------------------------------------------
    1. 800-Series                               (key: etk800)
       設定数: 29, 作者: BeamNG
    2. Aurata                                   (key: utv)
       設定数: 7, 作者: BeamNG
    3. BX-Series                                (key: bx)
       設定数: 36, 作者: BeamNG
    4. Barstow                                  (key: barstow)
       設定数: 21, 作者: BeamNG
    5. Bastion                                  (key: bastion)
       設定数: 20, 作者: BeamNG
    6. Bluebuck                                 (key: bluebuck)
       設定数: 34, 作者: BeamNG
    7. Bolide                                   (key: bolide)
       設定数: 21, 作者: BeamNG
    8. Covet                                    (key: covet)
       設定数: 40, 作者: BeamNG
    9. FCV                                      (key: vivace)
       設定数: 35, 作者: BeamNG
   10. Grand Marshal               

### 詳細設定

In [5]:
# ============================================================================
# 特定の車両keyの詳細設定を表示する関数
# ============================================================================
def show_vehicle_configs(vehicle_key):
    """指定した車両の利用可能な設定を全て表示"""
    if vehicle_key not in vehicles_data:
        print(f"\nエラー: '{vehicle_key}' は存在しません")
        return
    
    vehicle_info = vehicles_data[vehicle_key]
    configs = vehicle_info['configurations']
    
    print(f"\n{'='*80}")
    print(f"{vehicle_info['name']} ({vehicle_key})")
    print(f"{'='*80}")
    print(f"タイプ: {vehicle_info['type']}")
    print(f"作者: {vehicle_info['author']}")
    print(f"設定数: {len(configs)}\n")
    
    for idx, (config_key, config_data) in enumerate(sorted(configs.items()), 1):
        print(f"{idx:3d}. {config_data['name']:<60} config='{config_key}'")
    
    print(f"\n{'='*80}")
    print("使用例:")
    print(f"vehicle = Vehicle('my_vehicle', model='{vehicle_key}', config='設定key')")
    print(f"{'='*80}\n")

In [6]:
# ============================================================================
# インタラクティブに選択できるようにする
# ============================================================================
def interactive_config_viewer():
    """対話的に車両を選択して設定を表示"""
    print("\n" + "="*80)
    print("車両key を入力してください (終了する場合は 'q')")
    print("="*80)
    
    while True:
        vehicle_key = input("\n車両key: ").strip()
        
        if vehicle_key.lower() == 'q':
            print("終了します")
            break
        
        if not vehicle_key:
            continue
        
        show_vehicle_configs(vehicle_key)

interactive_config_viewer()


車両key を入力してください (終了する場合は 'q')

Aurata (utv)
タイプ: Car
作者: BeamNG
設定数: 7

  1. Aurata Base (CVT)                                            config='base'
  2. Aurata Custom - Asphalt (CVT)                                config='custom'
  3. Aurata Junior (CVT)                                          config='junior'
  4. Aurata Sport (CVT)                                           config='plus'
  5. Aurata Race (CVT)                                            config='race'
  6. Aurata Sport Turbo (CVT)                                     config='turbo'
  7. Aurata Wild (CVT)                                            config='wild'

使用例:
vehicle = Vehicle('my_vehicle', model='utv', config='設定key')

終了します


### Vehicle Spwan

In [7]:
vehicle = Vehicle('ego_vehicle', model='utv', config='race', licence='ego_vehicle')

scenario = Scenario('c1', 'LiDAR_demo', description='Spanning the map with a LiDAR sensor')

# Add the vehicle to the scenario with the specified initial position and orientation  
scenario.add_vehicle(vehicle,  
  pos=(3819.65, -5113.19, 852.37),  # Initial position (x, y, z)  
  rot_quat=(0.0, 0.0, 0.35836795, 0.93358043)  # Initial orientation as a quaternion (x, y, z, w)  
)


scenario.make(bng)
bng.settings.set_deterministic(60)
bng.load_scenario(scenario)
bng.ui.hide_hud()
bng.scenario.start()

2025-11-15 15:39:30,840 |INFO     |beamngpy.BeamNGpy             |Loaded map.
2025-11-15 15:39:34,466 |INFO     |beamngpy.Vehicle              |Vehicle ego_vehicle connected to simulation.
2025-11-15 15:39:34,467 |INFO     |beamngpy.BeamNGpy             |Attempting to connect to vehicle ego_vehicle
2025-11-15 15:39:36,919 |INFO     |beamngpy.BeamNGpy             |Successfully connected to BeamNG.tech.
2025-11-15 15:39:36,920 |INFO     |beamngpy.BeamNGpy             |Successfully connected to vehicle ego_vehicle.
2025-11-15 15:39:36,922 |INFO     |beamngpy.Scenario             |Connected to scenario: LiDAR_demo
2025-11-15 15:39:36,924 |INFO     |beamngpy.BeamNGpy             |Starting scenario.
